# 05 — The survey's two secondary questions

**A. Math calibration (1.5B).** Does calibrating GPTQ on math text (GSM8K train) fix math, cost facts,
or change the mix of errors? Compared with the main GPTQ (C4 calibration, same grid and settings) and RTN.

**B. Does FP16 confidence predict the *kind* of error?** Notebook 03 showed that low FP16 confidence predicts
*which* answers flip. Here: among the broken answers, does FP16 confidence differ between error types
(e.g. near-miss vs unrelated, conceptual vs execution)? Done for every size.

**Inputs (+ Add Input → Your Work → Notebooks):**
- 03 notebooks: `03_Flips` (1.5B, the version run with the math-calibrated model), `03_Flips_3B`, `03_Flips_0.5B`
- 04 notebooks: `04_judge` (1.5B facts), `04 Judge v2` (1.5B math), `04_Judge_1.5B_mathcal`, `04_judge_3B`, `04_Judge_0.5B`
- optional: `02_Quantize_1.5B_math` (for the perplexity line)

Missing inputs are skipped with a note. **Accelerator: None. Internet: not needed.**

**Saved:** `05_results.json`, `05_confidence_by_error_type.csv`.

In [ ]:
# 1. Settings: which file holds which labels
LABELS = {                    # (size, task) -> labels file from notebook 04
    ("1.5b", "triviaqa"): "04_labels.jsonl",                 # main study, facts
    ("1.5b", "gsm8k"):    "04_labels_math_v2.jsonl",         # main study, math (final judge)
    ("1.5b_mathcal", "triviaqa"): "04_labels_1.5b_mathcal.jsonl",
    ("1.5b_mathcal", "gsm8k"):    "04_labels_1.5b_mathcal.jsonl",
    ("3b", "triviaqa"): "04_labels_3b.jsonl",   ("3b", "gsm8k"): "04_labels_3b.jsonl",
    ("0.5b", "triviaqa"): "04_labels_0.5b.jsonl", ("0.5b", "gsm8k"): "04_labels_0.5b.jsonl",
}
FLIPS = {"1.5b": "03_harmful_flips.jsonl", "3b": "03_harmful_flips_3b.jsonl", "0.5b": "03_harmful_flips_0.5b.jsonl"}
RESULTS_15 = "03_results.json"
PPL_MATH = "02_qwen2.5-1.5b_math_results.json"
PPL_C4_15 = {"fp16": (9.65, 15.65), "rtn4": (10.74, 17.23), "gptq4": (10.19, 16.28)}   # 1.5B main study (WikiText2, C4)
NAMES = {"fp16": "FP16", "rtn4": "RTN", "gptq4": "GPTQ (C4)", "gptq4math": "GPTQ (math)"}
TASKN = {"gsm8k": "math", "triviaqa": "facts"}
MIN_N = 10            # error types with fewer broken answers are left out of the tests
N_BOOT = 2000
SEED = 0
OUT_DIR = "/kaggle/working"

In [ ]:
# 2. Load
import glob, json, os
import numpy as np, pandas as pd
from scipy.stats import chi2_contingency, kruskal, mannwhitneyu

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    return hits[0] if hits else None

def read_jsonl(path):
    out = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    out.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return out

flips, labels = {}, {}
for size, name in FLIPS.items():
    p = find(name)
    if p:
        df = pd.DataFrame(read_jsonl(p))
        df["key"] = df["task"] + "|" + df["model"] + "|" + df["id"].astype(str)
        flips[size] = df
        print(f"flips  {size:5s} {len(df):5d}  <- {p}")
    else:
        print(f"flips  {size:5s} MISSING ({name})")
for (size, task), name in LABELS.items():
    p = find(name)
    if p:
        lab = pd.DataFrame(read_jsonl(p))
        lab = lab[(lab["task"] == task) & lab["label"].notna()]
        labels[(size, task)] = lab.set_index("key")["label"]
        print(f"labels {size:12s} {task:8s} {len(lab):5d}  <- {p}")
    else:
        print(f"labels {size:12s} {task:8s} MISSING ({name})")
results = {"A_math_calibration": {}, "B_confidence_by_error_type": {}}

## A. Math calibration (1.5B): GPTQ calibrated on GSM8K train vs GPTQ calibrated on C4

In [ ]:
# 3. A — accuracy, broken answers and head-to-head (from notebook 03, 1.5B)
p = find(RESULTS_15)
r15 = json.load(open(p)) if p else None
if not r15 or "gptq4math" not in r15.get("models", []):
    print("03_results.json with the math-calibrated model not attached -> part A skipped")
else:
    rows = []
    for task in ["gsm8k", "triviaqa"]:
        for m in ["fp16", "rtn4", "gptq4", "gptq4math"]:
            a = r15["accuracy"][f"{m}_{task}"]
            row = {"task": TASKN[task], "model": NAMES[m], "accuracy %": round(100 * a["acc"], 1),
                   "95% CI": f"{100*a['ci95'][0]:.1f}–{100*a['ci95'][1]:.1f}"}
            if m != "fp16":
                f = r15["flips"][f"{m}_{task}"]
                row.update({"broken": f["harmful"], "repaired": f["repair"],
                            "broken % of FP16-right": round(100 * f["harmful_rate_of_fp16_right"], 1),
                            "change vs FP16 (pts)": round(100 * f["net_change"], 1)})
            rows.append(row)
    accA = pd.DataFrame(rows)
    display(accA)
    results["A_math_calibration"]["accuracy"] = rows
    rows = []
    for task in ["gsm8k", "triviaqa"]:
        for a, b in [("gptq4", "gptq4math"), ("rtn4", "gptq4math")]:
            h = r15["head_to_head"].get(f"{a}_vs_{b}_{task}")
            if h:
                rows.append({"task": TASKN[task], "pair": f"{NAMES[a]} vs {NAMES[b]}", "broken by both": h["broken_by_both"],
                             f"only {NAMES[a]}": h[f"only_{a}"], f"only {NAMES[b]}": h[f"only_{b}"],
                             "overlap (Jaccard)": round(h["overlap_jaccard"], 2), "McNemar p": h["mcnemar_p"]})
    display(pd.DataFrame(rows))
    results["A_math_calibration"]["head_to_head"] = rows
    p = find(PPL_MATH)
    if p:
        pm = json.load(open(p))
        ppl = {**PPL_C4_15, "gptq4math": (pm["gptq4math_wiki"], pm["gptq4math_c4"])}
        display(pd.DataFrame([{"model": NAMES[m], "WikiText2": round(w, 2), "C4": round(c, 2)} for m, (w, c) in ppl.items()]))
        results["A_math_calibration"]["perplexity"] = {m: {"wikitext2": w, "c4": c} for m, (w, c) in ppl.items()}

In [ ]:
# 4. A — do the error types change with math calibration? (chi-square, scoring_issue left out)
def mix_test(task, groups):
    """groups: list of (name, label Series). Returns the % table and a chi-square test."""
    rows = {name: s.value_counts() for name, s in groups if s is not None and len(s)}
    if len(rows) < 2:
        return None, None
    tab = pd.DataFrame(rows).T.fillna(0).astype(int)
    pct = (100 * tab.div(tab.sum(axis=1), axis=0)).round(1)
    real = tab.drop(columns=[c for c in ["scoring_issue"] if c in tab.columns])
    real = real.loc[:, real.sum() > 0]
    chi2, p, dof, exp = chi2_contingency(real)
    return pct.assign(total=tab.sum(axis=1)), {"chi2": chi2, "dof": dof, "p": p, "min_expected": float(exp.min())}

def labels_of(size, task, model):
    s = labels.get((size, task))
    if s is None:
        return None
    return s[s.index.str.startswith(f"{task}|{model}|")]

mixA = {}
for task in ["gsm8k", "triviaqa"]:
    math = labels_of("1.5b_mathcal", task, "gptq4math")
    for other in ["gptq4", "rtn4"]:
        pct, test = mix_test(task, [(NAMES[other], labels_of("1.5b", task, other)), (NAMES["gptq4math"], math)])
        if pct is None:
            print(f"{TASKN[task]}: labels missing -> skipped"); continue
        print(f"\n=== {TASKN[task]}: {NAMES['gptq4math']} vs {NAMES[other]} (% of each model's broken answers)")
        display(pct)
        print(f"chi-square = {test['chi2']:.2f}, dof = {test['dof']}, p = {test['p']:.4f}"
              + ("   (some expected counts < 5: read with care)" if test["min_expected"] < 5 else ""))
        mixA[f"{task}_gptq4math_vs_{other}"] = {"percent": pct.to_dict(), **test}
results["A_math_calibration"]["error_mix"] = mixA

## B. Does FP16 confidence predict the kind of error?

For every broken answer we know how confident FP16 was in its (correct) answer: the **lowest token
log-probability** (main, as in notebook 03) and the mean. Higher = more confident.
**Kruskal–Wallis**: do the error types differ in FP16 confidence at all? **Pairwise AUROC**: chance that a
broken answer of the first type had *higher* FP16 confidence than one of the second (0.5 = no difference).
Pre-chosen pairs: facts **near-miss vs unrelated**, math **execution vs conceptual**.

In [ ]:
# 5. B — FP16 confidence of each error type, per size and task (RTN and GPTQ pooled, then per method)
rng = np.random.default_rng(SEED)
PAIRS = {"triviaqa": ("near_miss", "unrelated"), "gsm8k": ("execution", "conceptual")}

def auc_pair(x, y):
    """P(a random x > a random y), ties count half, with a bootstrap 95% CI."""
    x, y = np.asarray(x), np.asarray(y)
    u = mannwhitneyu(x, y, alternative="two-sided")
    auc = u.statistic / (len(x) * len(y))
    boots = []
    for _ in range(N_BOOT):
        xb, yb = rng.choice(x, len(x)), rng.choice(y, len(y))
        boots.append(mannwhitneyu(xb, yb).statistic / (len(x) * len(y)))
    return auc, np.percentile(boots, 2.5), np.percentile(boots, 97.5), u.pvalue

table, tests = [], []
for size in ["0.5b", "1.5b", "3b"]:
    if size not in flips:
        continue
    for task in ["triviaqa", "gsm8k"]:
        lab = labels.get((size, task))
        if lab is None:
            continue
        df = flips[size]
        df = df[(df["task"] == task) & df["model"].isin(["rtn4", "gptq4"])].copy()
        df["label"] = df["key"].map(lab)
        df = df[df["label"].notna() & (df["label"] != "scoring_issue")]
        for scope, g in [("RTN + GPTQ", df), ("RTN", df[df.model == "rtn4"]), ("GPTQ", df[df.model == "gptq4"])]:
            types = [t for t, n in g["label"].value_counts().items() if n >= MIN_N]
            for t in sorted(g["label"].unique()):
                s = g[g.label == t]
                table.append({"size": size, "task": TASKN[task], "methods": scope, "error type": t, "n": len(s),
                              "median min log-prob": round(s["fp16_min_logprob"].median(), 3),
                              "median mean log-prob": round(s["fp16_mean_logprob"].median(), 3)})
            entry = {"size": size, "task": TASKN[task], "methods": scope, "types tested": ", ".join(types)}
            if len(types) >= 2:
                h, p = kruskal(*[g[g.label == t]["fp16_min_logprob"] for t in types])
                entry.update({"Kruskal-Wallis H": round(h, 2), "p (all types)": p})
            a, b = PAIRS[task]
            if (g.label == a).sum() >= MIN_N and (g.label == b).sum() >= MIN_N:
                auc, lo, hi, pp = auc_pair(g[g.label == a]["fp16_min_logprob"], g[g.label == b]["fp16_min_logprob"])
                entry.update({"pair": f"{a} > {b}", "AUROC": round(auc, 3), "95% CI": f"{lo:.3f}–{hi:.3f}", "p (pair)": pp})
            tests.append(entry)

conf = pd.DataFrame(table)
tests = pd.DataFrame(tests)
pd.set_option("display.max_rows", 200)
print("FP16 confidence (lowest token log-prob; closer to 0 = more confident) by error type")
display(conf[conf.methods == "RTN + GPTQ"].drop(columns="methods"))
print("\nTests")
display(tests)
results["B_confidence_by_error_type"] = {"by_type": conf.to_dict("records"), "tests": tests.to_dict("records")}

In [ ]:
# 6. Save
def clean(o):
    if isinstance(o, dict):  return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o
json.dump(clean(results), open(f"{OUT_DIR}/05_results.json", "w"), indent=1)
conf.to_csv(f"{OUT_DIR}/05_confidence_by_error_type.csv", index=False)
print("saved: 05_results.json, 05_confidence_by_error_type.csv")